# 10 — Comparación final de modelos

Evalúa y compara todos los modelos entrenados en notebooks 07-09.

| Modelo | Notebook |
|--------|----------|
| Regresión Logística (baseline) | 07 |
| SVM | 08 |
| Random Forest | 08 |
| XGBoost | 09 |
| LightGBM | 09 |

## Métricas objetivo
| Métrica | Umbral |
|---------|--------|
| `recall_critico` | > 0.85 |
| `precision_critico` | > 0.70 |
| `f1_macro` | > 0.65 |

In [ ]:
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from sonalert.config import FIGURES_DIR, MODELS_DIR, PROCESSED_DATA_DIR, PROJ_ROOT

warnings.filterwarnings("ignore")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
CLASS_NAMES = ["bajo", "medio", "alto", "critico"]
CRITICO = 3

THRESHOLDS = {
    "recall_critico":    0.85,
    "precision_critico": 0.70,
    "f1_macro":          0.65,
}

In [ ]:
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI")
MLFLOW_ACTIVE = bool(MLFLOW_URI)

if MLFLOW_ACTIVE:
    import mlflow
    import mlflow.sklearn

    mlflow.set_tracking_uri(MLFLOW_URI)
    mlflow.set_experiment("comparacion_modelos")
    print(f"[MLflow] activo: {MLFLOW_URI}")
else:
    print("[MLflow] inactivo — resultados locales.")

## 2. Cargar splits

In [ ]:
X_val     = pd.read_parquet(PROCESSED_DATA_DIR / "X_val.parquet")
X_holdout = pd.read_parquet(PROCESSED_DATA_DIR / "X_holdout.parquet")
y_val     = pd.read_parquet(PROCESSED_DATA_DIR / "y_val.parquet")["y"]
y_holdout = pd.read_parquet(PROCESSED_DATA_DIR / "y_holdout.parquet")["y"]

print(f"X_val:     {X_val.shape}")
print(f"X_holdout: {X_holdout.shape}")
print(f"\nClass balance val:     {dict(y_val.value_counts(normalize=True).sort_index().round(3))}")
print(f"Class balance holdout: {dict(y_holdout.value_counts(normalize=True).sort_index().round(3))}")

## 3. Cargar modelos

In [ ]:
MODEL_FILES = {
    "LogReg":  MODELS_DIR / "logreg_best.joblib",
    "SVM":     MODELS_DIR / "svm_best.joblib",
    "RF":      MODELS_DIR / "rf_best.joblib",
    "XGB":     MODELS_DIR / "xgb_best.joblib",
    "LGBM":    MODELS_DIR / "lgbm_best.joblib",
}

models = {}
for name, path in MODEL_FILES.items():
    if path.exists():
        models[name] = joblib.load(path)
        print(f"  [OK] {name}: {path.name}")
    else:
        print(f"  [SKIP] {name}: archivo no encontrado ({path.name})")

print(f"\nModelos cargados: {list(models.keys())}")

## 4. Evaluación

In [ ]:
def evaluate_model(model, X, y) -> dict:
    y_pred = model.predict(X)
    return {
        "recall_critico":    recall_score(y, y_pred, labels=[CRITICO], average="macro", zero_division=0),
        "precision_critico": precision_score(y, y_pred, labels=[CRITICO], average="macro", zero_division=0),
        "f1_macro":          f1_score(y, y_pred, average="macro", zero_division=0),
        "accuracy":          float((y_pred == y).mean()),
        "y_pred":            y_pred,
    }


results = {}
for name, model in models.items():
    val_m     = evaluate_model(model, X_val, y_val)
    holdout_m = evaluate_model(model, X_holdout, y_holdout)
    results[name] = {"val": val_m, "holdout": holdout_m}
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"  VAL     recall_critico={val_m['recall_critico']:.3f}  "
          f"precision_critico={val_m['precision_critico']:.3f}  "
          f"f1_macro={val_m['f1_macro']:.3f}")
    print(f"  HOLDOUT recall_critico={holdout_m['recall_critico']:.3f}  "
          f"precision_critico={holdout_m['precision_critico']:.3f}  "
          f"f1_macro={holdout_m['f1_macro']:.3f}")

## 5. Tabla comparativa

In [ ]:
METRICS = ["recall_critico", "precision_critico", "f1_macro", "accuracy"]

rows = []
for name, res in results.items():
    for split in ["val", "holdout"]:
        row = {"modelo": name, "split": split}
        row.update({m: res[split][m] for m in METRICS})
        rows.append(row)

df_results = pd.DataFrame(rows)

df_holdout = (
    df_results[df_results["split"] == "holdout"]
    .set_index("modelo")
    [METRICS]
    .sort_values("recall_critico", ascending=False)
)

# Marcar si supera umbrales
def style_threshold(val, col):
    threshold = THRESHOLDS.get(col)
    if threshold is None:
        return ""
    color = "#2ECC71" if val >= threshold else "#E74C3C"
    return f"background-color: {color}; color: white"

print("=== HOLDOUT — métricas por modelo ===")
display(
    df_holdout.style
    .format("{:.3f}")
    .apply(lambda col: [style_threshold(v, col.name) for v in col])
)

## 6. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ["#2E86AB", "#E63946", "#2ECC71", "#F39C12", "#9B59B6"]

for ax, metric in zip(axes, ["recall_critico", "precision_critico", "f1_macro"]):
    vals_val     = [results[n]["val"][metric]     for n in results]
    vals_holdout = [results[n]["holdout"][metric] for n in results]
    x = np.arange(len(results))
    w = 0.35

    ax.bar(x - w/2, vals_val,     w, label="val",     color="#2E86AB", alpha=0.85)
    ax.bar(x + w/2, vals_holdout, w, label="holdout", color="#E63946", alpha=0.85)

    if metric in THRESHOLDS:
        ax.axhline(THRESHOLDS[metric], color="black", linestyle="--", linewidth=1.2,
                   label=f"umbral ({THRESHOLDS[metric]})")

    ax.set_xticks(x)
    ax.set_xticklabels(list(results.keys()), rotation=20, ha="right")
    ax.set_title(metric.replace("_", " "))
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Comparación de modelos — val vs holdout", fontsize=13, fontweight="bold")
plt.tight_layout()
bar_path = FIGURES_DIR / "10_comparacion_barras.png"
plt.savefig(bar_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {bar_path.relative_to(PROJ_ROOT)}")

In [ ]:
heatmap_data = df_holdout[METRICS[:3]].T  # 3 métricas × N modelos

fig, ax = plt.subplots(figsize=(max(6, len(results) * 1.5), 3.5))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".3f", cmap="RdYlGn",
    vmin=0, vmax=1,
    linewidths=0.5, linecolor="white",
    ax=ax,
)
ax.set_title("Heatmap métricas — holdout", fontsize=12)
plt.tight_layout()
heatmap_path = FIGURES_DIR / "10_comparacion_heatmap.png"
plt.savefig(heatmap_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {heatmap_path.relative_to(PROJ_ROOT)}")

In [ ]:
top3 = df_holdout["recall_critico"].nlargest(3).index.tolist()

fig, axes = plt.subplots(1, len(top3), figsize=(5 * len(top3), 4.5))
if len(top3) == 1:
    axes = [axes]

for ax, name in zip(axes, top3):
    y_pred = results[name]["holdout"]["y_pred"]
    cm = confusion_matrix(y_holdout, y_pred, labels=[0, 1, 2, 3])
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d"
    )
    rc = results[name]["holdout"]["recall_critico"]
    ax.set_title(f"{name} — recall_critico={rc:.3f}")

plt.suptitle("Top 3 modelos — Confusion Matrix (holdout)", fontsize=12, fontweight="bold")
plt.tight_layout()
cm_path = FIGURES_DIR / "10_top3_confusion.png"
plt.savefig(cm_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {cm_path.relative_to(PROJ_ROOT)}")

## 7. Selección del modelo campeón

In [ ]:
# Criterio: recall_critico primario, f1_macro secundario
df_rank = df_holdout.copy()
df_rank["cumple_recall"]    = df_rank["recall_critico"]    >= THRESHOLDS["recall_critico"]
df_rank["cumple_precision"] = df_rank["precision_critico"] >= THRESHOLDS["precision_critico"]
df_rank["cumple_f1"]        = df_rank["f1_macro"]          >= THRESHOLDS["f1_macro"]
df_rank["cumple_todos"]     = df_rank[["cumple_recall", "cumple_precision", "cumple_f1"]].all(axis=1)

campeon = df_rank.sort_values(
    ["cumple_todos", "recall_critico", "f1_macro"],
    ascending=[False, False, False]
).index[0]

print(f"\n{'='*50}")
print(f"  MODELO CAMPEÓN: {campeon}")
print(f"{'='*50}")
print(df_rank.loc[campeon, METRICS + ["cumple_todos"]].to_string())
print(f"\nCumple todos los umbrales: {df_rank.loc[campeon, 'cumple_todos']}")

print("\n--- Modelos que cumplen TODOS los umbrales ---")
cumplen = df_rank[df_rank["cumple_todos"]]
if cumplen.empty:
    print("  Ningún modelo supera los 3 umbrales simultáneamente.")
    print("  Campeón elegido por recall_critico máximo.")
else:
    print(cumplen[METRICS].to_string())

In [ ]:
print("\nClassification report completo — campeón en holdout:")
y_pred_campeon = results[campeon]["holdout"]["y_pred"]
print(classification_report(y_holdout, y_pred_campeon, target_names=CLASS_NAMES, digits=3, zero_division=0))

## 8. Registro en MLflow

In [ ]:
if MLFLOW_ACTIVE:
    for name, res in results.items():
        with mlflow.start_run(run_name=f"compare_{name}"):
            mlflow.set_tags({
                "model_family": name,
                "stage":        "model_comparison",
                "is_champion":  str(name == campeon),
            })
            for split in ["val", "holdout"]:
                for metric in METRICS:
                    mlflow.log_metric(f"{split}_{metric}", res[split][metric])
    print(f"[MLflow] {len(results)} runs registrados. Campeón: {campeon}")
else:
    print(f"[MLflow] inactivo. Campeón: {campeon}")

## 9. Resumen final

In [ ]:
print("=" * 55)
print(" RESUMEN FINAL — HOLDOUT")
print("=" * 55)
print(df_holdout[METRICS].round(3).to_string())
print("\nUmbrales objetivo:")
for k, v in THRESHOLDS.items():
    print(f"  {k}: > {v}")
print(f"\nModelo campeón: {campeon}")
print("=" * 55)
print("\nFiguras generadas:")
for p in [bar_path, heatmap_path, cm_path]:
    print(f"  {p.relative_to(PROJ_ROOT)}")